### LLM 프레임워크

In [12]:
from langchain_ollama import ChatOllama
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")

# LangChain + watsonx 사용

In [13]:
watson_llm = ChatWatsonx(
    model_id = "ibm/granite-4-h-small",
    url = watsonx_ai_url,
    api_key = apikey,
    project_id = project_id,
    max_tokens = 2000
)

response = watson_llm.invoke("Hello")
print(response.content)

Hello! How may I assist you today?


In [14]:
qwen_llm = ChatOllama(model="qwen3.5:4b")

response = qwen_llm.invoke("Hello")
print(response.content)

Hello! How can I assist you today?


In [17]:
exaone_llm = ChatOllama(model="exaone3.5:2.4b")

response = exaone_llm.invoke("현재 대통령은 누구지?")
print(response.content)

제가 제공하는 정보에 따르면, LG AI 연구원의 모델로서 실시간 세계 뉴스나 특정 지역의 정치 정보에 직접 접근할 수는 없습니다. 하지만, 가장 최근의 업데이트를 제공하기 위해 저는 일반적인 지식을 바탕으로 답변 드리겠습니다. 정확한 대통령 정보를 확인하려면 해당 국가의 공식 웹사이트나 신뢰할 수 있는 국제 뉴스 출처를 참조하시는 것이 가장 좋습니다. 한국의 경우, 대한민국 대통령의 이름은 현재 윤석열 대통령입니다 (2022년부터 재임 중). 하지만 특정 시점의 업데이트가 필요하다면 최신 출처를 확인하시는 것이 중요합니다.


### PromptTemplate
- 하나의 문자열 프롬프트 생성

In [20]:
# ① 기본 생성(이 방법 보다는 아래 방법이 더 많이 사용됨)
template = PromptTemplate(
    input_variables=["topic", "level"],
    template="{level} 수준으로 {topic}을 설명해줘. 예시 포함.",
)

# ② 프롬프트 미리 확인 (LLM 호출 전 디버깅에 유용)
prompt_text = template.format(topic="재귀함수", level="초급자")
print(prompt_text)
# 초급자 수준으로 재귀함수을 설명해줘. 예시 포함.

# ③ from_template 단축 문법 (더 많이 씀)
template2 = PromptTemplate.from_template(
    "{language} 코드로 {task}를 구현해줘. 주석 포함."
)
# 위 방법처럼 굳이 명시하지 않아도 인식되는 걸 볼 수 있음
print(template2.input_variables)  # ['language', 'task']
template2.format(language="python", task="재귀함수")

초급자 수준으로 재귀함수을 설명해줘. 예시 포함.
['language', 'task']


'python 코드로 재귀함수를 구현해줘. 주석 포함.'

In [22]:
template = PromptTemplate.from_template("""
    다음 질문에 답변하시오.

    질문:
    {question}
    """)

formatted_prompt = template.format(question="SQL Injection이란?")

response = watson_llm.invoke(formatted_prompt)
print(response.content)

SQL Injection은 웹 애플리케이션에 대한 일종의 공격 기법입니다. 이 공격은 악의적으로 작성된 SQL 문을 웹 애플리케이션의 입력 필드에 삽입함으로써 데이터베이스에서 원하지 않는 작업을 수행하거나 데이터베이스 내의 데이터를 노출시키려는 시도를 포함합니다. 이 공격은 일반적으로 웹 애플리케이션이 사용자 입력을 충분히 검증하지 않을 때 발생합니다.

예를 들어, 웹 애플리케이션에서 로그인을 위해 사용자 이름과 비밀번호를 입력받는 필드가 있다고 가정해 보겠습니다. 애플리케이션은 이 입력을 데이터베이스에 대한 질의문(query)로 변환하여 사용자의 신원을 확인합니다. 하지만 만약 이 필드에 아래와 같이 악의적인 SQL 코드가 삽입되면,

```
' OR '1'='1
```

그리고 이것이 검증되지 않은 채로 데이터베이스에 전달된다면, 어플리케이션은 원래의 로그인을 확인하는 질의문 대신 악의적인 코드를 실행할 수 있게 됩니다. 이로 인해 데이터베이스의 민감한 정보가 노출되거나, 악의적인 행위자가 데이터베이스에서 원하지 않는 작업을 수행할 수 있게 됩니다.

따라서 SQL Injection 공격을 방지하기 위해서는 웹 애플리케이션에서 사용자 입력을 충분히 검증하고, 데이터베이스에 전달되는 모든 질의문을 파라미터화하는 등의 방어 기법을 사용해야 합니다.


#### ChatPromptTemplate
- 채팅 메시지 형식
- from_template()
- from_messages()

In [23]:
template = ChatPromptTemplate.from_messages(
[
    ("system", "당신은 {role} 전문가입니다. {language} 로 답변하세요."),
    ("human", "{question}"),
]
)

formatted_prompt = template.format(role="파이썬", language="한국어", question="리스트 컴프리헨션이란?")
response = watson_llm.invoke(formatted_prompt)
print(response.content)

리스트 컴프리헨션(list comprehension)은 파이썬에서 리스트를 생성하는 간결하고 표현력이 뛰어난 방법입니다. 리스트 컴프리헨션을 사용하면, 반복문(for), 조건문(if) 등을 이용하여 리스트를 생성할 수 있고, 이를 한 줄로 표현할 수 있습니다.

아래의 예시에서는 1부터 5까지의 숫자를 제곱하여 리스트로 생성한 것입니다:

```python
squares = [x**2 for x in range(1, 6)]
print(squares)
```

출력 결과는 다음과 같습니다:

```
[1, 4, 9, 16, 25]
```

리스트 컴프리헨션은 이보다 더욱 복잡한 구문을 작성할 수도 있습니다. 예를 들어, 아래의 코드는 홀수인 숫자만을 리스트로 생성합니다:

```python
odd_squares = [x**2 for x in range(1, 11) if x%2 == 1]
print(odd_squares)
```

출력 결과는 다음과 같습니다:

```
[1, 9, 25, 49, 81]
```

연습을 통해 리스트 컴프리헨션을 익히고 사용하면, 코드 작성이 더욱 효율적이고 간결하게 될 것입니다.


In [24]:
template = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 {role} 전문가입니다. {language} 로 답변하세요."),
        ("human", "{question}"),
    ]
)

formatted_prompt = template.format(
    role="파이썬", language="한국어", question="리스트 컴프리헨션이란?"
)

exaone_llm = ChatOllama(model="exaone3.5:2.4b")

response = exaone_llm.invoke(formatted_prompt)
print(response.content)

리스트 컴프리헨션은 파이썬에서 간결하고 효과적으로 리스트를 생성하기 위한 기법입니다. 일반적인 반복문 대신 간결한 표현식으로 리스트 요소를 만드는 방법입니다. 기본 구조는 다음과 같습니다:

```python
새로운_리스트 = [표현식(요소) for 요소 in 원본_반복가능_object if 조건(요소)]
```

여기서:
- `표현식(요소)`: 각 요소를 변환하거나 가공하는 부분입니다.
- `원본_반복가능_object`: 리스트 컴프리헨션이 적용될 반복 가능한 객체 (예: 리스트, 범위 등)입니다.
- `조건(요소)` (선택적): 요소를 필터링하는 조건을 추가할 수 있습니다. 생략되면 모든 요소가 포함됩니다.

예를 들어, 숫자 리스트에서 짝수만 추출하는 경우:
```python
even_numbers = [x for x in range(10) if x % 2 == 0]
```
이 코드는 `range(10)`에서 짝수만 선택하여 `[0, 2, 4, 6, 8]` 리스트를 생성합니다. 리스트 컴프리헨션은 코드를 더 읽기 쉽고 효율적으로 만들어 줍니다.


#### LCEL 파이프라인
- LCEL (LangChain Expression Language) : | 연산자로 컴포넌트를 연결하는 파이프라인 문법
    - ex) prompt | llm | parser => 왼쪽에서 오른쪽으로 데이터가 흐르게 됨
- invoke() : 단일 동기 호출 / streaming : 스트리밍 형태 / batch() : 여러 입력을 병렬 처리

In [25]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 {role} 전문가입니다."),
        ("human", "{question}"),
    ]
)

exaone_llm =  ChatOllama(model="exaone3.5:2.4b")
chain = prompt | exaone_llm

response = chain.invoke({
    "role" : "보안",
    "question" : "XSS란?"
})
print(response.content)

XSS는 **Cross-Site Scripting**의 약자로, **웹 애플리케이션 보안 취약점 중 하나**입니다. 쉽게 말해 **악성 스크립트를 웹 페이지에 삽입하여 다른 사용자의 브라우저에서 실행되도록 하는 공격 기법**입니다.

**어떻게 작동하는지 자세히 살펴보겠습니다:**

1. **악성 코드 삽입:** 공격자는 웹사이트에 악의적인 JavaScript 코드를 hidden 변수, URL 쿼리 문자열, 쿠키 등을 통해 위장하여 주입합니다. 예를 들어, 댓글 섹션에 사용자 입력을 처리하지 않고 직접 표시하는 경우 XSS 취약점이 발생할 수 있습니다.

2. **사용자 세션 하이재킹:** 주입된 스크립트는 공격자가 사용자의 쿠키 정보를 탈취하거나 세션을 탈취하여 사용자 계정에 무단 접근할 수 있도록 합니다.

3. **피싱 공격:** 악성 스크립트는 가짜 로그인 페이지를 생성하여 사용자 정보를 입력받거나, 사용자가 의도하지 않은 행동을 하도록 유도할 수 있습니다 (예: 개인 정보 유출, 계정 탈취).

4. **사이트 악성 코드 배포:** 공격자는 XSS를 통해 자신의 악성 코드를 설치하여 사용자들에게 피해를 입힐 수 있습니다.

**XSS 공격의 주요 유형:**

* **Stored XSS:** 악성 코드가 웹 서버에 영구적으로 저장되어 모든 사용자가 열람할 수 있는 유형입니다 (예: 게시판 댓글).
* **Reflected XSS:** 사용자 입력이 공격 URL에 포함되어 웹 서버에서 반사되어 실행되는 유형입니다 (예: 가짜 검색 결과).
* **DOM-Based XSS:** 클라이언트 측 자바스크립트 코드를 통해 발생하며, 서버에서 검증 과정을 거치지 않고 악성 코드를 실행합니다.

**예방 및 대응 방안:**

* **입력 검증 및 이스케이프:** 사용자 입력을 엄격하게 검증하고 HTML 특수 문자를 이스케이프하여 스크립트 실행을 방지합니다.
* **Content Security Policy (CSP) 설정:** 웹 페이지에서 실행 가능한 자원의 출처

In [31]:
for chunk in chain.stream({
    "role" : "보안",
    "question" : "XSS란?"
}):
    print(chunk, end='', flush=True)


content='X' additional_kwargs={} response_metadata={} id='lc_run--019e66f9-c307-74a3-9a70-4bc5513ba82b' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]content='SS' additional_kwargs={} response_metadata={} id='lc_run--019e66f9-c307-74a3-9a70-4bc5513ba82b' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]content=' (' additional_kwargs={} response_metadata={} id='lc_run--019e66f9-c307-74a3-9a70-4bc5513ba82b' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]content='Cross' additional_kwargs={} response_metadata={} id='lc_run--019e66f9-c307-74a3-9a70-4bc5513ba82b' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]content='-' additional_kwargs={} response_metadata={} id='lc_run--019e66f9-c307-74a3-9a70-4bc5513ba82b' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]content='Site' additional_kwargs={} response_metadata={} id='lc_run--019e66f9-c307-74a3-9a70-4bc5513ba82b' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]content=' Script' additional_kwargs={

In [35]:
# ③ batch — 여러 입력 병렬 처리
results = chain.batch(
    [
        {"role": "user", "question": "리스트란?"},
        {"role": "user", "question": "딕셔너리란?"},
        {"role": "user", "question": "튜플이란?"},
    ]
)

for result in results:
    print(result)

content='리스트(List)는 컴퓨터 과학 및 프로그래밍에서 매우 중요한 데이터 구조 중 하나입니다. 리스트는 다음과 같은 특징을 가지고 있습니다:\n\n1. **순서 있는 저장**: 리스트는 요소들을 특정 순서대로 저장합니다. 일반적으로 인덱스(0부터 시작하는 위치 번호)를 통해 각 요소에 접근할 수 있습니다. 예를 들어, 정수 리스트 `[1, 2, 3]`에서 `index(0)`은 `1`을 반환합니다.\n\n2. **동적 크기**: 리스트는 필요에 따라 요소를 추가하거나 제거할 수 있어 동적으로 크기를 조정할 수 있습니다. 이는 프로그래밍에서 유연한 데이터 관리에 매우 유용합니다.\n\n3. **다양한 유형의 요소**: 리스트는 단일 유형의 요소(예: 정수 리스트)뿐만 아니라 다양한 유형의 요소(예: 정수, 문자열, 튜플 등)를 포함할 수 있습니다. 때로는 복잡한 데이터 구조(객체 등)를 포함하기도 합니다.\n\n4. **접근 방식**:\n   - **인덱스 기반 접근**: 특정 위치의 요소에 직접 접근합니다. 예: `myList[0]`은 리스트의 첫 번째 요소를 반환합니다.\n   - **슬라이싱**: 범위를 지정하여 부분 리스트를 추출할 수 있습니다. 예: `myList[1:3]`은 인덱스 1부터 3까지의 요소를 포함하는 새로운 리스트를 반환합니다.\n\n5. **표준 라이브러리**: 대부분의 프로그래밍 언어(Python, Java, C++ 등)는 리스트를 내장 라이브러리 형태로 제공하여 쉽게 사용할 수 있게 합니다. 예를 들어, Python에서는 `list()` 함수를 사용하거나 `[]` 구문으로 리스트를 생성합니다.\n\n리스트는 배열과 유사하지만 몇 가지 중요한 차이점이 있습니다:\n- **배열**: 고정된 크기를 가지고 있으며, 일반적으로 특정 타입의 데이터만 저장할 수 있습니다 (예: 정수 배열).\n- **리스트**: 동적으로 크기 조정 가능하며, 다양한 타입의 요소를 포함할 수 있습니다.\n\n이러한 특성 때문에 리스트는 배열에서 확장된

In [44]:
from rich import print
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

# StrOutputParser

str_chain = ChatPromptTemplate.from_template(
    "{topic}을 한 문장으로 설명해줘" 
) | watson_llm | StrOutputParser()

result = str_chain.invoke({"topic" : "머신러닝"})

print(result)

# JSONOutputParser

json_chain = ChatPromptTemplate.from_messages([
    ('system', 'JSON 형식으로만 응답'),
    ('human', '{text}의 감정을 분석해서 {{"sentiment" : "...", "score" : "0.0" }} 형태로'),
]) | watson_llm | JsonOutputParser()

result = json_chain.invoke({"text" : "오늘 정말 행복한 하루였어요"})

print("[bold blue]>>> JsonOutputParser 결과[/bold blue]")
print(f"[bold]전체 결과:[/bold] {result}")
print(f"[bold]감정(sentiment):[/bold] {result['sentiment']}")
print(f"[bold]점수(score):[/bold] {result['score']}")

# PydanticOutputParser
# 반환 타입 : 객체
# 입력 데이터 타입 검증, 타입 자동 변환
# 에러 메시지가 정확하게 나옴, 코드 간결, FastAPI 호환, json 변환 쉬움


머신러닝은 컴퓨터가 경험을 통해 학습하고 개선할 수 있는 AI 기술로, 데이터로부터 패턴을 인식하고 예측할 수 있는 
능력을 습득합니다.

>>> JsonOutputParser 결과

전체 결과: {'sentiment': '긍정적', 'score': '0.8'}

감정(sentiment): 긍정적

점수(score): 0.8

# PydanticOutputParser
    
    # 반환 타입 : 객체
    # 입력 데이터 타입 검증, 타입 자동 변환
    # 에러 메시지가 정확하게 나옴, 코드 간결, FastAPI 호환, json 변환 쉬움

In [46]:
from pydantic import BaseModel


class User(BaseModel):
    name: str
    age: int

user = User(name="John", age=30)
print(user)
print(type(user.age))

User(name='John', age=30)

<class 'int'>

In [66]:
from langchain_core.output_parsers import PydanticOutputParser
from typing import Literal
from pydantic import Field
from rich.console import Console
from rich.panel import Panel
from rich.text import Text

class SentimentResult(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(..., description="감정 상태")
    score: float = Field(..., ge=0.0, le=1.0, description="감정 강도")
    reason: str = Field(..., description="판단 근거 한 문장")
    keywords: list[str] = Field(..., description="핵심 키워드 3개 이내")

pydantic_parser = PydanticOutputParser(pydantic_object=SentimentResult)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 감정 분석 전문가입니다. 반드시 아래 형식으로만 응답하세요.\n\n{format_instructions}",
        ),
        ("human", "다음 텍스트의 감정을 분석해라: {text}"),
    ]
).partial(format_instructions=pydantic_parser.get_format_instructions())

texts = [
    "이 제품 정말 별롭니다. 돈 낭비 였습니다.",
    "그냥 평범한 하루였어요. 특별한 건 없습니다.",
    "와, 생각보다 훨씬 맛있었어요! 강력 추천합니다.",
    "서비스가 너무 불친절하고 음식도 늦게 나왔어요."
]

chain = prompt | watson_llm | pydantic_parser
results = chain.batch([{'text' : t} for t in texts])

console = Console()

emoji = {"positive": "😀", "negative": "😞", "neutral": "😐"}
color = {"positive": "green", "negative": "red", "neutral": "yellow"}

for text, r in zip(texts, results):
    c = color[r.sentiment]
    console.rule(
        f"[{c}]{emoji[r.sentiment]} {r.sentiment}  ({r.score:.2f})[/]", style=c
    )
    console.print(f"[bold]입력[/]    {text}")
    console.print(f"[bold]이유[/]    {r.reason}")
    console.print(f"[bold]키워드[/]  {', '.join(r.keywords)}")
    console.print()

─────────────────────────────────────────────── 😞 negative  (1.00) ───────────────────────────────────────────────

입력    이 제품 정말 별롭니다. 돈 낭비 였습니다.

이유    이 제품이 기대에 못 미치고 돈 낭비라는 표현이 포함되어 있어 부정적인 감정으로 판단됩니다.

키워드  별롭습니다, 돈 낭비

─────────────────────────────────────────────── 😐 neutral  (0.00) ────────────────────────────────────────────────

...

키워드  맛있었어요, 강력, 추천

─────────────────────────────────────────────── 😞 negative  (0.80) ───────────────────────────────────────────────

입력    서비스가 너무 불친절하고 음식도 늦게 나왔어요.

이유    텍스트에서 서비스에 대한 불만과 음식 제공 지연을 언급하며 부정적인 경험을 전달합니다.

키워드  불친절, 늦게, 음식

In [67]:
# 카테고리 분류 추가
class AdvancedSentiment(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    score: float = Field(ge=0.0, le=1.0)
    category: Literal["product", "service", "food", "general"] = Field(
        description="리뷰 카테고리"
    )
    summary: str = Field(description="한 줄 요약")
    action: str = Field(description="비즈니스 조치 방향")


# 체인은 동일, 스키마만 교체
adv_parser = PydanticOutputParser(pydantic_object=AdvancedSentiment)
adv_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 CX(고객경험) 분석가입니다.\n{format_instructions}"),
        ("human", "{text}"),
    ]
).partial(format_instructions=adv_parser.get_format_instructions())

adv_chain = adv_prompt | ChatOllama(model="exaone3.5:2.4b", temperature=0) | adv_parser
r = adv_chain.invoke({"text": "배송은 빠른데 포장이 너무 허술해요"})
print(f"카테고리: {r.category} | 조치: {r.action}")

카테고리: product | 조치: 포장 개선 및 품질 관리 강화 필요